# 09 — Real Anonymized Speech: Content Evaluation

## Objective

Run the existing content-preservation framework on **real original/anonymized speech pairs** generated with an official VoicePrivacy Challenge 2026 baseline.

```
Original audio + anonymized audio
              ↓
          Faster-Whisper
              ↓
     original / anonymized transcripts
              ↓
    ┌─────────┼──────────┐
    ↓         ↓          ↓
Similarity   NLI   Semantic alignment → TPI
```

This notebook orchestrates the existing metrics. It does not redefine them.

## Experimental setup

- VoicePrivacy Challenge 2026 Track 1, LibriSpeech dev subset.
- Official B2 McAdams anonymization.
- Initial size: approximately 100 paired utterances.
- Same ASR configuration for original and anonymized audio.
- Content metrics remain separate; no arbitrary combined loss yet.

The pair manifest must contain `pair_id`, `original_path`, and `anonymized_path`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "vpc2026"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PAIR_MANIFEST = DATA_DIR / "pairs_100.csv"
TRANSCRIPT_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_transcripts.csv"
METRICS_OUTPUT = RESULTS_DIR / "vpc2026_b2_100_content_metrics.csv"

print("Project root:", PROJECT_ROOT)
print("Pair manifest:", PAIR_MANIFEST)

In [ ]:
if not PAIR_MANIFEST.exists():
    raise FileNotFoundError(
        f"Missing {PAIR_MANIFEST}. Create the 100-pair manifest after running VPC B2."
    )

pairs_df = pd.read_csv(PAIR_MANIFEST)
required = {"pair_id", "original_path", "anonymized_path"}
missing = required - set(pairs_df.columns)
if missing:
    raise ValueError(f"Missing manifest columns: {sorted(missing)}")

pairs_df = pairs_df.head(100).copy()

for col in ["original_path", "anonymized_path"]:
    pairs_df[col] = pairs_df[col].map(lambda p: str(Path(p).expanduser().resolve()))

missing_original = [p for p in pairs_df.original_path if not Path(p).exists()]
missing_anonymized = [p for p in pairs_df.anonymized_path if not Path(p).exists()]

if missing_original or missing_anonymized:
    raise FileNotFoundError(
        f"Missing audio files: {len(missing_original)} original, "
        f"{len(missing_anonymized)} anonymized"
    )

print("Pairs loaded:", len(pairs_df))
display(pairs_df.head())

## 1. ASR

Use the same Faster-Whisper model and decoding configuration for both sides. Keep the raw ASR transcripts before applying any content metric.

In [ ]:
from faster_whisper import WhisperModel

ASR_MODEL_NAME = "base"
ASR_DEVICE = "cpu"
ASR_COMPUTE_TYPE = "int8"
ASR_LANGUAGE = "en"

asr_model = WhisperModel(
    ASR_MODEL_NAME,
    device=ASR_DEVICE,
    compute_type=ASR_COMPUTE_TYPE,
)

def transcribe_audio(audio_path):
    segments, info = asr_model.transcribe(
        str(audio_path),
        language=ASR_LANGUAGE,
    )
    return " ".join(
        segment.text.strip()
        for segment in segments
        if segment.text.strip()
    )

def transcribe_pairs(df):
    rows = []
    for _, row in df.iterrows():
        out = row.to_dict()
        out["original_transcript"] = transcribe_audio(row.original_path)
        out["anonymized_transcript"] = transcribe_audio(row.anonymized_path)
        rows.append(out)
    return pd.DataFrame(rows)

In [ ]:
if TRANSCRIPT_OUTPUT.exists():
    transcript_df = pd.read_csv(TRANSCRIPT_OUTPUT)
    print("Loaded cached transcripts:", len(transcript_df))
else:
    transcript_df = transcribe_pairs(pairs_df)
    transcript_df.to_csv(TRANSCRIPT_OUTPUT, index=False)
    print("Transcribed pairs:", len(transcript_df))

display(transcript_df[["pair_id", "original_transcript", "anonymized_transcript"]].head(10))

## 2. Load the existing semantic-alignment + TPI implementation

The implementation is already in notebook 07e. To avoid maintaining a second copy, this notebook executes only the implementation cells needed for real-data evaluation and skips the controlled examples.

In [ ]:
import nbformat

METRIC_NOTEBOOK = PROJECT_ROOT / "notebooks" / "07e_semantic_alignment_TPI.ipynb"
if not METRIC_NOTEBOOK.exists():
    raise FileNotFoundError(METRIC_NOTEBOOK)

nb = nbformat.read(METRIC_NOTEBOOK, as_version=4)

metric_cell_indices = [2, 3, 5, 7, 9, 11, 12, 14, 16, 25]
metric_globals = globals()

for idx in metric_cell_indices:
    source = "".join(nb.cells[idx].source)
    exec(compile(source, f"07e_cell_{idx}", "exec"), metric_globals)

print("Loaded 07e metric implementation.")
print("SIMILARITY_THRESHOLD:", SIMILARITY_THRESHOLD)
print("TPI_STRATEGY:", TPI_STRATEGY, "N:", N)

## 3. Semantic similarity

The selected model from notebook 05c is `paraphrase-mpnet-base-v2`. Use the same model for both transcripts.

In [ ]:
from sentence_transformers import SentenceTransformer

SIM_MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"
sim_model = SentenceTransformer(SIM_MODEL_NAME)

original_embeddings = sim_model.encode(
    transcript_df["original_transcript"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)
anonymized_embeddings = sim_model.encode(
    transcript_df["anonymized_transcript"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
)

transcript_df["semantic_similarity"] = np.sum(
    original_embeddings * anonymized_embeddings,
    axis=1,
)

print("Semantic similarity computed for", len(transcript_df), "pairs.")

## 4. NLI

The exact NLI model/configuration from notebook 06b should be wired here. This cell intentionally does not silently substitute a different model.

In [ ]:
# TODO: copy the exact selected model and scoring code from notebook 06b.
NLI_MODEL_NAME = None

if NLI_MODEL_NAME is None:
    print("NLI is not wired yet. Run semantic similarity + alignment/TPI first.")
else:
    print("Load the selected NLI model here using the exact 06b configuration.")

## 5. Semantic alignment + TPI on real pairs

In [ ]:
metric_rows = []

for _, row in transcript_df.iterrows():
    result = evaluate_content_pair(
        row.original_transcript,
        row.anonymized_transcript,
        similarity_threshold=SIMILARITY_THRESHOLD,
        tpi_strategy=TPI_STRATEGY,
        N=N,
    )

    metric_rows.append({
        "pair_id": row.pair_id,
        "TPI": result["TPI"],
        "L_TPI": result["L_TPI"],
        "TIC_original": result["tpi"]["TIC_original"],
        "information_lost": result["tpi"]["information_lost"],
        "n_affected_terms": len(result["alignment"]["affected_original_indices"]),
        "affected_terms": ", ".join(result["alignment"]["affected_original_terms"]),
    })

metrics_df = pd.DataFrame(metric_rows)
metrics_df = metrics_df.merge(
    transcript_df[["pair_id", "semantic_similarity"]],
    on="pair_id",
    how="left",
)

metrics_df.to_csv(METRICS_OUTPUT, index=False)
display(metrics_df.head(10))

## 6. First diagnostics

Do not combine the metrics into one loss yet. First inspect distributions, correlations, and disagreements.

In [ ]:
print("Number of pairs:", len(metrics_df))
display(metrics_df[[
    "semantic_similarity", "TPI", "L_TPI", "n_affected_terms"
]].describe())

print("Correlation:")
display(metrics_df[["semantic_similarity", "TPI"]].corr())

In [ ]:
low_tpi_high_sim = metrics_df[
    (metrics_df["TPI"] < metrics_df["TPI"].quantile(0.25)) &
    (metrics_df["semantic_similarity"] > metrics_df["semantic_similarity"].quantile(0.75))
]

print("Potential high-similarity / low-TPI cases:", len(low_tpi_high_sim))

display(
    transcript_df.merge(
        low_tpi_high_sim[["pair_id", "TPI", "semantic_similarity"]],
        on="pair_id"
    )[[
        "pair_id",
        "original_transcript",
        "anonymized_transcript",
        "semantic_similarity",
        "TPI"
    ]].head(10)
)

## 7. Interpretation

The first experiment is not intended to prove that TPI is valid or invalid.

Questions to answer:

1. Does semantic similarity remain high when information is changed or lost?
2. Does TPI identify changes that semantic similarity misses?
3. Where does NLI disagree with TPI?
4. Does semantic alignment fail because of ASR noise, paraphrasing, reordering, insertions, or deletions?
5. Which real examples should become new controlled tests?

Only after this analysis should we decide whether to refine the content framework or scale the experiment.